In [1]:
# ==============================================================================
# Cell 1: Environment Setup, Libraries, & Dataset Path Configuration
# ==============================================================================

# Core data manipulation & numeric computing
import os
import sys
import gc
import time
import glob
from pathlib import Path

import pandas as pd
import numpy as np

# Visualization & Exploratory Data Analysis (EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# Memory monitoring & progress tracking
import psutil
from tqdm.auto import tqdm

In [2]:
# ------------------------------------------------------------------------------
# 1. Display & Visualization Options
# ------------------------------------------------------------------------------
# Prevent Pandas column truncation during inspection
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Plotting configuration
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

def print_memory_usage():
    """Prints the current process RAM utilization."""
    process = psutil.Process(os.getpid())
    mem_mb = process.memory_info().rss / (1024 * 1024)
    print(f"[System Memory] Current RAM usage: {mem_mb:.2f} MB")


In [3]:
# ------------------------------------------------------------------------------
# 2. Path Resolution (Resolves root whether run from root or notebooks dir)
# ------------------------------------------------------------------------------
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parents[1]
elif cwd.name == "ml":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DATA_DIR = PROJECT_ROOT / "ml" / "data" / "raw" / "CSE-CIC-IDS2018"
INTERIM_DATA_DIR = PROJECT_ROOT / "ml" / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "ml" / "data" / "processed"

# Ensure output directories exist
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
# ------------------------------------------------------------------------------
# 3. Discover & Inventory Raw CSV Files
# ------------------------------------------------------------------------------
csv_files = sorted(list(RAW_DATA_DIR.glob("*.csv")))

print("=" * 75)
print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Dir: {RAW_DATA_DIR}")
print(f"Discovered {len(csv_files)} CSV files in CSE-CIC-IDS2018:")
print("=" * 75)

total_size_bytes = 0
file_inventory = []

for idx, file_path in enumerate(csv_files, 1):
    size_mb = file_path.stat().st_size / (1024 * 1024)
    total_size_bytes += file_path.stat().st_size
    file_inventory.append({
        "Index": idx,
        "Filename": file_path.name,
        "Size_MB": round(size_mb, 2),
        "Path": file_path
    })
    print(f" [{idx:02d}] {file_path.name:<52} ({size_mb:>8.2f} MB)")

print("-" * 75)
print(f"Total Dataset Size: {total_size_bytes / (1024**3):.2f} GB")
print_memory_usage()
print("=" * 75)

# Convert inventory to DataFrame for easy indexing in subsequent cells
df_inventory = pd.DataFrame(file_inventory)


Project Root: C:\Users\himma\Desktop\Projects\AI project\SentinelAI
Raw Data Dir: C:\Users\himma\Desktop\Projects\AI project\SentinelAI\ml\data\raw\CSE-CIC-IDS2018
Discovered 10 CSV files in CSE-CIC-IDS2018:
 [01] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv      (  336.04 MB)
 [02] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv      (  318.26 MB)
 [03] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv      (  365.11 MB)
 [04] Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv    ( 3867.08 MB)
 [05] Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv    (  102.85 MB)
 [06] Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv    (  358.53 MB)
 [07] Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv    (  364.91 MB)
 [08] Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv   (  341.63 MB)
 [09] Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv   (  313.66 MB)
 [10] Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv   (  199.56 MB)
--------------------------------------------------------------

In [5]:
# ------------------------------------------------------------------------------
# 4. Helper Function: Memory-Optimized CSV Loader
# ------------------------------------------------------------------------------
def load_csv_file(file_path, nrows=None, use_cols=None):
    """
    Loads a CSV file with automatic whitespace stripping on column headers
    and garbage collection to manage memory safely.
    """
    print(f"Loading: {Path(file_path).name} (nrows={nrows})...")
    df = pd.read_csv(file_path, nrows=nrows, usecols=use_cols, low_memory=False)
    # Strip leading/trailing whitespaces in column names (standard for CIC datasets)
    df.columns = df.columns.str.strip()
    return df


In [6]:
# ==============================================================================
# Cell 2: Inspect Rows, Columns, and Preview Top Lines
# ==============================================================================

# Select a file to inspect (e.g., the first file in the inventory)
target_file = csv_files[0]
print(f"Inspecting file: {target_file.name}")

# Load a sample (e.g., 10,000 rows for fast inspection, or set nrows=None for full file)
df_sample = load_csv_file(target_file, nrows=10000)

# 1. Number of rows and columns
num_rows, num_cols = df_sample.shape
print(f"\n📊 Dataset Dimensions:")
print(f" • Number of Rows:    {num_rows:,}")
print(f" • Number of Columns: {num_cols}")

# 2. Preview the top 5 rows
print("\n🔍 Top 5 Rows:")
display(df_sample.head(5))

# 3. List of all column names
print("\n📋 Column Names:")
print(list(df_sample.columns))


Inspecting file: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Loading: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv (nrows=10000)...

📊 Dataset Dimensions:
 • Number of Rows:    10,000
 • Number of Columns: 80

🔍 Top 5 Rows:


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443,6,02/03/2018 08:47:38,141385,9,7,553,3773,202,0,61.4444,87.5344,1460,0,539.0000,655.4329,30597.3052,113.1662,9425.6667,19069.1168,73403.0000,1.0000,141385.0000,17673.1250,23965.3233,73403.0000,22.0000,51417.0000,8569.5000,13036.8908,31525.0000,1.0000,0,0,0,0,192,152,63.6560,49.5102,0,1460,254.4706,474.7130,225352.3897,0,0,1,1,0,0,0,1,0,270.3750,61.4444,539.0000,0,0,0,0,0,0,9,553,7,3773,8192,119,4,20,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,Benign
1,49684,6,02/03/2018 08:47:38,281,2,1,38,0,38,0,19.0000,26.8701,0,0,0.0000,0.0000,135231.3167,10676.1566,140.5000,174.6554,264.0000,17.0000,281.0000,281.0000,0.0000,281.0000,281.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,0,0,0,40,20,7117.4377,3558.7189,0,38,19.0000,21.9393,481.3333,0,1,0,0,1,0,0,0,0,25.3333,19.0000,0.0000,0,0,0,0,0,0,2,38,1,0,123,0,0,20,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,Benign
2,443,6,02/03/2018 08:47:40,279824,11,15,1086,10527,385,0,98.7273,129.3925,1460,0,701.8000,636.3142,41501.0864,92.9155,11192.9600,24379.4483,112589.0000,1.0000,279728.0000,27972.8000,36167.7403,112589.0000,94.0000,258924.0000,18494.5714,36356.5037,133669.0000,1.0000,0,0,0,0,232,312,39.3104,53.6051,0,1460,430.1111,566.2342,320621.1795,0,0,1,1,0,0,0,1,1,446.6538,98.7273,701.8000,0,0,0,0,0,0,11,1086,15,10527,8192,1047,5,20,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,Benign
3,443,6,02/03/2018 08:47:40,132,2,0,0,0,0,0,0.0000,0.0000,0,0,0.0000,0.0000,0.0000,15151.5151,132.0000,0.0000,132.0000,132.0000,132.0000,132.0000,0.0000,132.0000,132.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,40,0,15151.5151,0.0000,0,0,0.0000,0.0000,0.0000,0,0,0,0,1,0,0,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,2,0,0,0,256,-1,0,20,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,Benign
4,443,6,02/03/2018 08:47:41,274016,9,13,1285,6141,517,0,142.7778,183.8877,1460,0,472.3846,611.1805,27100.6073,80.2873,13048.3810,26311.6270,114077.0000,1.0000,273946.0000,34243.2500,37996.5655,114077.0000,201.0000,252994.0000,21082.8333,39075.7382,135611.0000,1.0000,0,0,0,0,192,272,32.8448,47.4425,0,1460,322.8696,497.2548,247262.3004,0,0,1,1,0,0,0,1,1,337.5455,142.7778,472.3846,0,0,0,0,0,0,9,1285,13,6141,8192,1047,5,20,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,Benign



📋 Column Names:
['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk R

All 10 files Info

In [7]:
# ==============================================================================
# Cell 3: Row & Column Counts Across All 10 CSV Files
# ==============================================================================

summary_data = []

print("Analyzing row and column counts across all files...")
for file_path in tqdm(csv_files, desc="Inspecting CSVs"):
    # Read just the header to get column count
    header_df = pd.read_csv(file_path, nrows=0)
    col_count = len(header_df.columns)
    
    # Fast row counting without loading full dataset into memory
    with open(file_path, "rb") as f:
        row_count = sum(1 for _ in f) - 1  # subtract 1 for header
    
    summary_data.append({
        "File Name": file_path.name,
        "Rows": row_count,
        "Columns": col_count,
        "Size (MB)": round(file_path.stat().st_size / (1024 * 1024), 2)
    })

df_summary = pd.DataFrame(summary_data)

# Display the summary table
display(df_summary)

print(f"\nTotal Rows Across All Files: {df_summary['Rows'].sum():,}")
print(f"Total Columns per File:      {df_summary['Columns'].iloc[0]}")


Analyzing row and column counts across all files...


Inspecting CSVs:   0%|          | 0/10 [00:00<?, ?it/s]

,File Name,Rows,Columns,Size (MB)
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,80,336.0400
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,318.2600
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,365.1100
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,84,3867.0800
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,80,102.8500
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,358.5300
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,364.9100
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,80,341.6300
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,80,313.6600
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,80,199.5600



Total Rows Across All Files: 16,233,002
Total Columns per File:      80


In [9]:
df_summary.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   File Name  10 non-null     str    
 1   Rows       10 non-null     int64  
 2   Columns    10 non-null     int64  
 3   Size (MB)  10 non-null     float64
dtypes: float64(1), int64(2), str(1)
memory usage: 939.0 bytes


In [10]:
# ==============================================================================
# Cell 3: Cross-File Consistency Audit & Full Info Inspector
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Cross-File Consistency Audit (Columns, Dtypes, & Corrupt Rows)
# ------------------------------------------------------------------------------
print("=" * 80)
print("AUDITING ALL 10 CSV FILES FOR COLUMN & DTYPE CONSISTENCY")
print("=" * 80)

audit_results = []
all_file_columns = {}
all_file_dtypes = {}

# Use the first 80-column file as the baseline reference
ref_file = csv_files[0]
ref_df_header = pd.read_csv(ref_file, nrows=10, low_memory=False)
ref_cols = [c.strip() for c in ref_df_header.columns]

for file_path in tqdm(csv_files, desc="Auditing files"):
    # Read a sample to inspect columns and data types
    df_chunk = pd.read_csv(file_path, nrows=5000, low_memory=False)
    raw_cols = list(df_chunk.columns)
    stripped_cols = [c.strip() for c in raw_cols]
    
    # Store columns & datatypes
    all_file_columns[file_path.name] = stripped_cols
    all_file_dtypes[file_path.name] = {c.strip(): str(t) for c, t in df_chunk.dtypes.items()}
    
    # Check for difference with baseline
    extra_cols = set(stripped_cols) - set(ref_cols)
    missing_cols = set(ref_cols) - set(stripped_cols)
    
    # Check if raw headers had leading/trailing spaces
    has_whitespace = any(c != c.strip() for c in raw_cols)
    
    # Check for repeated header rows inside data (corrupt rows)
    label_col = [c for c in df_chunk.columns if 'label' in c.lower()][0]
    has_repeated_headers = (df_chunk[label_col].astype(str) == 'Label').any()
    
    # Unique labels found in sample
    labels_found = df_chunk[label_col].dropna().unique().tolist()
    if 'Label' in labels_found:
        labels_found.remove('Label')
    
    audit_results.append({
        "Filename": file_path.name,
        "Col Count": len(stripped_cols),
        "Extra Cols": list(extra_cols) if extra_cols else "None",
        "Missing Cols": list(missing_cols) if missing_cols else "None",
        "Whitespace in Header": has_whitespace,
        "Has Corrupt Header Rows": has_repeated_headers,
        "Sample Labels": labels_found
    })

df_audit = pd.DataFrame(audit_results)
display(df_audit[["Filename", "Col Count", "Extra Cols", "Has Corrupt Header Rows", "Sample Labels"]])


# ------------------------------------------------------------------------------
# 2. Detailed Info Function for ANY File
# ------------------------------------------------------------------------------
def inspect_file_full_info(file_index_or_path, sample_rows=50000):
    """
    Displays a comprehensive summary of any chosen file:
    - Data dimensions & memory usage
    - Column-by-column datatypes, missing values, and unique counts
    - Label / class distribution
    """
    if isinstance(file_index_or_path, int):
        file_path = csv_files[file_index_or_path]
    else:
        file_path = Path(file_index_or_path)
        
    print("=" * 80)
    print(f"FULL DATASET INSPECTION: {file_path.name}")
    print(f"Sample size: {sample_rows:,} rows (for safe memory usage)")
    print("=" * 80)
    
    # Load sample with cleaned column names
    df = pd.read_csv(file_path, nrows=sample_rows, low_memory=False)
    df.columns = df.columns.str.strip()
    
    # Remove any repeated header rows if present
    if 'Label' in df.columns:
        df = df[df['Label'] != 'Label'].copy()
        
    # 1. High-level Summary
    mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"Shape in sample: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Sample RAM Footprint: {mem_mb:.2f} MB")
    print(f"Dtype breakdown:\n{df.dtypes.value_counts()}\n")
    
    # 2. Column-by-column detailed profile
    col_info = []
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        n_unique = df[col].nunique()
        sample_val = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
        
        col_info.append({
            "Column Name": col,
            "Dtype": str(df[col].dtype),
            "Null Count": null_count,
            "Null %": f"{null_pct:.2f}%",
            "Unique Values": n_unique,
            "Sample Value": sample_val
        })
        
    df_col_info = pd.DataFrame(col_info)
    print("📋 Column Profiles (First 20 columns displayed):")
    display(df_col_info.head(20))
    
    # 3. Label Breakdown
    if 'Label' in df.columns:
        print("\n🎯 Label / Target Distribution:")
        label_dist = df['Label'].value_counts(dropna=False).reset_index()
        label_dist.columns = ['Class', 'Count']
        label_dist['Percentage'] = (label_dist['Count'] / len(df)) * 100
        display(label_dist)
        
    return df_col_info

# Example: Inspect file 0 (or change index to inspect any of the 10 files)
# inspect_file_full_info(0)
# inspect_file_full_info(3)  # Inspects the 84-column Thuesday file


AUDITING ALL 10 CSV FILES FOR COLUMN & DTYPE CONSISTENCY


Auditing files:   0%|          | 0/10 [00:00<?, ?it/s]

,Filename,Col Count,Extra Cols,Has Corrupt Header Rows,Sample Labels
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,80,None,False,"[Benign, Bot]"
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,80,None,False,"[Benign, DoS attacks-SlowHTTPTest]"
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,80,None,False,"[Benign, Brute Force -Web, Brute Force -XSS, S..."
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,84,"[Flow ID, Dst IP, Src Port, Src IP]",False,"[Benign, DDoS attacks-LOIC-HTTP]"
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,80,None,True,[Benign]
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,80,None,False,"[Benign, DoS attacks-GoldenEye]"
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,80,None,False,"[Benign, Brute Force -Web, Brute Force -XSS, S..."
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,80,None,False,"[Benign, FTP-BruteForce]"
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,80,None,False,"[Benign, DDOS attack-LOIC-UDP, DDOS attack-HOIC]"
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,80,None,False,[Benign]


In [11]:
# ==============================================================================
# Cell 4: Deep Inspection of 'Thuesday-20-02-2018' (84 Columns)
# ==============================================================================

# 1. Target file path
thuesday_file = RAW_DATA_DIR / "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv"
print(f"Target File: {thuesday_file.name}")
print(f"File Size:   {thuesday_file.stat().st_size / (1024**3):.2f} GB\n")

# 2. Load top rows (loading 1,000 rows ensures representative dtypes & null checks)
# Since the full file is ~4 GB, reading a sample is instantaneous and memory-safe
df_thuesday = pd.read_csv(thuesday_file, nrows=1000, low_memory=False)
df_thuesday.columns = df_thuesday.columns.str.strip()

# ------------------------------------------------------------------------------
# Part 1: Top 5 Rows Preview
# ------------------------------------------------------------------------------
print("=" * 85)
print(f"🔍 TOP 5 ROWS PREVIEW (Shape: {df_thuesday.shape[0]} rows x {df_thuesday.shape[1]} columns)")
print("=" * 85)

# Ensure all 84 columns are visible horizontally in Jupyter
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    display(df_thuesday.head(5))

# ------------------------------------------------------------------------------
# Part 2: Complete Column Info Table (All 84 Columns)
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("📋 COMPLETE INFORMATION FOR ALL 84 COLUMNS")
print("=" * 85)

extra_cols = {'Flow ID', 'Src IP', 'Src Port', 'Dst IP'}

col_info_list = []
for idx, col in enumerate(df_thuesday.columns, 1):
    col_info_list.append({
        "No.": idx,
        "Column Name": col,
        "Dtype": str(df_thuesday[col].dtype),
        "Non-Null Count": df_thuesday[col].notnull().sum(),
        "Null Count": df_thuesday[col].isnull().sum(),
        "Unique Values": df_thuesday[col].nunique(),
        "Sample Value (Row 0)": df_thuesday[col].iloc[0],
        "Notes": "⚠️ Extra Column (Only in this file)" if col in extra_cols else "Standard (In all 10 files)"
    })

df_all_columns = pd.DataFrame(col_info_list)

# Display all 84 rows without pagination truncation
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 50):
    display(df_all_columns)

# ------------------------------------------------------------------------------
# Part 3: Quick Standard df.info() Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 85)
print("📊 STANDARD DF.INFO() SUMMARY")
print("=" * 85)
df_thuesday.info(verbose=True, show_counts=True)


Target File: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
File Size:   3.78 GB

🔍 TOP 5 ROWS PREVIEW (Shape: 1000 rows x 84 columns)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,172.31.69.25-94.231.103.172-22-45498-6,94.231.103.172,45498,172.31.69.25,22,6,20/02/2018 08:34:07,888751,11,11,1249,1969,736,0,113.5455,220.8961,976,0,179.0000,364.1865,3620.8117,24.7538,42321.4762,47851.7358,101609.0000,14.0000,888751.0000,88875.1000,49295.7355,140273.0000,14.0000,788197.0000,78819.7000,55863.1880,139285,72,0,0,0,0,360,360,12.3769,12.3769,0,976,139.9130,290.6339,84468.0830,0,0,0,1,0,0,0,0,1,146.2727,113.5455,179.0000,0,0,0,0,0,0,11,1249,11,1969,14600,233,7,32,0,0,0,0,0.0000,0.0000,0.0000,0.0000,Benign
1,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:33:22,112642816,3,0,0,0,0,0,0.0000,0.0000,0,0,0.0000,0.0000,0.0000,0.0266,56300000.0000,7.0711,56300000.0000,56300000.0000,113000000.0000,56300000.0000,7.0711,56300000.0000,56300000.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0.0266,0.0000,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0,0,0,0,56300000.0000,7.0711,56300000.0000,56300000.0000,Benign
2,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:36:11,112642712,3,0,0,0,0,0,0.0000,0.0000,0,0,0.0000,0.0000,0.0000,0.0266,56300000.0000,18.3848,56300000.0000,56300000.0000,113000000.0000,56300000.0000,18.3848,56300000.0000,56300000.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0.0266,0.0000,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0,0,0,0,56300000.0000,18.3848,56300000.0000,56300000.0000,Benign
3,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:39:00,112642648,3,0,0,0,0,0,0.0000,0.0000,0,0,0.0000,0.0000,0.0000,0.0266,56300000.0000,5.6569,56300000.0000,56300000.0000,113000000.0000,56300000.0000,5.6569,56300000.0000,56300000.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0.0266,0.0000,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0,0,0,0,56300000.0000,5.6569,56300000.0000,56300000.0000,Benign
4,8.0.6.4-8.6.0.1-0-0-0,8.6.0.1,0,8.0.6.4,0,0,20/02/2018 08:41:49,112642702,3,0,0,0,0,0,0.0000,0.0000,0,0,0.0000,0.0000,0.0000,0.0266,56300000.0000,65.0538,56300000.0000,56300000.0000,113000000.0000,56300000.0000,65.0538,56300000.0000,56300000.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0.0266,0.0000,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0,0,0,0,0,0,3,0,0,0,-1,-1,0,0,0,0,0,0,56300000.0000,65.0538,56300000.0000,56300000.0000,Benign



📋 COMPLETE INFORMATION FOR ALL 84 COLUMNS


,No.,Column Name,Dtype,Non-Null Count,Null Count,Unique Values,Sample Value (Row 0),Notes
0,1,Flow ID,str,1000,0,959,172.31.69.25-94.231.103.172-22-45498-6,⚠️ Extra Column (Only in this file)
1,2,Src IP,str,1000,0,26,94.231.103.172,⚠️ Extra Column (Only in this file)
2,3,Src Port,int64,1000,0,586,45498,⚠️ Extra Column (Only in this file)
3,4,Dst IP,str,1000,0,4,172.31.69.25,⚠️ Extra Column (Only in this file)
4,5,Dst Port,int64,1000,0,5,22,Standard (In all 10 files)
5,6,Protocol,int64,1000,0,3,6,Standard (In all 10 files)
6,7,Timestamp,str,1000,0,80,20/02/2018 08:34:07,Standard (In all 10 files)
7,8,Flow Duration,int64,1000,0,1000,888751,Standard (In all 10 files)
8,9,Tot Fwd Pkts,int64,1000,0,14,11,Standard (In all 10 files)
9,10,Tot Bwd Pkts,int64,1000,0,9,11,Standard (In all 10 files)



📊 STANDARD DF.INFO() SUMMARY
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 84 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Flow ID            1000 non-null   str    
 1   Src IP             1000 non-null   str    
 2   Src Port           1000 non-null   int64  
 3   Dst IP             1000 non-null   str    
 4   Dst Port           1000 non-null   int64  
 5   Protocol           1000 non-null   int64  
 6   Timestamp          1000 non-null   str    
 7   Flow Duration      1000 non-null   int64  
 8   Tot Fwd Pkts       1000 non-null   int64  
 9   Tot Bwd Pkts       1000 non-null   int64  
 10  TotLen Fwd Pkts    1000 non-null   int64  
 11  TotLen Bwd Pkts    1000 non-null   int64  
 12  Fwd Pkt Len Max    1000 non-null   int64  
 13  Fwd Pkt Len Min    1000 non-null   int64  
 14  Fwd Pkt Len Mean   1000 non-null   float64
 15  Fwd Pkt Len Std    1000 non-null   float64
 16  Bwd Pk

In [2]:
DATA_DIR = "../data/raw/CSE-CIC-IDS2018"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
quality_records = []
col_records = []
for fpath in files:
    fname = os.path.basename(fpath)
    total_rows = 0
    total_nulls = 0
    total_infs = 0
    header_repeats = 0
    null_cols = {}
    inf_cols = {}
    
    for chunk in pd.read_csv(fpath, chunksize=300000, low_memory=False):
        total_rows += len(chunk)
        
        # Check repeated header rows
        first_col = chunk.columns[0]
        header_repeats += int((chunk[first_col].astype(str) == first_col).sum())
        
        # Null values
        null_counts = chunk.isna().sum()
        for col, cnt in null_counts[null_counts > 0].items():
            null_cols[col] = null_cols.get(col, 0) + int(cnt)
            total_nulls += int(cnt)
            
        # Infinite values
        for col in ['Flow Byts/s', 'Flow Pkts/s']:
            if col in chunk.columns:
                if pd.api.types.is_numeric_dtype(chunk[col]):
                    infs = int(np.isinf(chunk[col]).sum())
                else:
                    infs = int(chunk[col].astype(str).str.strip().isin(['Infinity', 'inf', '-Infinity', '-inf']).sum())
                if infs > 0:
                    inf_cols[col] = inf_cols.get(col, 0) + infs
                    total_infs += infs
    quality_records.append({
        'File': fname,
        'Rows': total_rows,
        'Missing Cells': total_nulls,
        'Infinite Cells': total_infs,
        'Header Repeats': header_repeats
    })
    
    for col in set(null_cols) | set(inf_cols):
        col_records.append({
            'File': fname,
            'Column': col,
            'Missing': null_cols.get(col, 0),
            'Infinite': inf_cols.get(col, 0)
        })
df_quality = pd.DataFrame(quality_records)
df_cols = pd.DataFrame(col_records)
print("=== Summary by File ===")
display(df_quality)
print("\n=== Summary by Affected Column ===")
display(df_cols)


=== Summary by File ===


,File,Rows,Missing Cells,Infinite Cells,Header Repeats
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,2558,5542,0
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,0,0,1
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3754,7662,0
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,36767,82139,0
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,1834,4004,25
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,4921,11133,0
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3569,7651,0
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,2277,5371,0
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,0,0,0
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,4041,8297,33



=== Summary by Affected Column ===


,File,Column,Missing,Infinite
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,2558,1492
1,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,4050
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,3754,1954
3,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,5708
4,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,36767,22686
5,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,59453
6,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,1834,1085
7,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,2919
8,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,4921,3106
9,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,8027


In [4]:
# ==============================================================================
# Cell: Check Duplicate Rows Across All Files (Memory-Safe)
# ==============================================================================
import os
import glob
import time
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw/CSE-CIC-IDS2018"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

duplicate_results = []
grand_total_rows = 0
grand_total_dups = 0

print(f"Scanning duplicate rows across {len(files)} files...\n")

for idx, fpath in enumerate(files, 1):
    fname = os.path.basename(fpath)
    t0 = time.time()
    seen_hashes = set()
    file_rows = 0
    file_dups = 0
    
    # Process in 300,000-row chunks to prevent Out-Of-Memory errors
    for chunk in pd.read_csv(fpath, chunksize=300000, low_memory=False):
        file_rows += len(chunk)
        
        # Compute 64-bit row hash
        hashes = pd.util.hash_pandas_object(chunk, index=False)
        
        # Identify duplicates within current chunk and across previous chunks
        dups_within = hashes.duplicated()
        dups_cross = hashes.isin(seen_hashes)
        is_dup = dups_within | dups_cross
        
        file_dups += int(is_dup.sum())
        seen_hashes.update(hashes[~is_dup])
        
    elapsed = time.time() - t0
    dup_pct = (file_dups / file_rows * 100) if file_rows > 0 else 0
    grand_total_rows += file_rows
    grand_total_dups += file_dups
    
    print(f"[{idx:02d}/10] {fname}: {file_dups:,} duplicates ({dup_pct:.2f}%) / {file_rows:,} rows in {elapsed:.1f}s")
    
    duplicate_results.append({
        'File': fname,
        'Total Rows': file_rows,
        'Duplicate Rows': file_dups,
        'Duplicate %': round(dup_pct, 3),
        'Unique Rows': file_rows - file_dups
    })

# Format into DataFrame summary
df_duplicates = pd.DataFrame(duplicate_results)

# Add Grand Total row
total_pct = (grand_total_dups / grand_total_rows * 100) if grand_total_rows > 0 else 0
df_total = pd.DataFrame([{
    'File': 'GRAND TOTAL',
    'Total Rows': grand_total_rows,
    'Duplicate Rows': grand_total_dups,
    'Duplicate %': round(total_pct, 3),
    'Unique Rows': grand_total_rows - grand_total_dups
}])
df_summary = pd.concat([df_duplicates, df_total], ignore_index=True)

print("\n" + "="*80)
print(f"TOTAL DUPLICATES ACROSS ALL FILES: {grand_total_dups:,} ({total_pct:.2f}%)")
print("="*80)
display(df_summary)


Scanning duplicate rows across 10 files...

[01/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 5,459 duplicates (0.52%) / 1,048,575 rows in 13.9s
[02/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 137,229 duplicates (13.09%) / 1,048,575 rows in 15.7s
[03/10] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 2,614 duplicates (0.25%) / 1,048,575 rows in 15.8s
[04/10] Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 2 duplicates (0.00%) / 7,948,748 rows in 222.9s
[05/10] Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 97 duplicates (0.03%) / 331,125 rows in 11.5s
[06/10] Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 2,421 duplicates (0.23%) / 1,048,575 rows in 15.4s
[07/10] Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 3,278 duplicates (0.31%) / 1,048,575 rows in 14.8s
[08/10] Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 225,628 duplicates (21.52%) / 1,048,575 rows in 14.1s
[09/10] Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 17,557 duplicates (1.67

,File,Total Rows,Duplicate Rows,Duplicate %,Unique Rows
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,5459,0.521,1043116
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,137229,13.087,911346
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,2614,0.249,1045961
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,2,0.000,7948746
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,97,0.029,331028
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,2421,0.231,1046154
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3278,0.313,1045297
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,225628,21.518,822947
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,17557,1.674,1031018
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,6121,0.998,606983


In [5]:
# Load the pre-computed quality report
df_report = pd.read_csv("../data/reports/CSE-CIC-IDS2018/data_quality_by_file.csv")
display(df_report)


,File,Rows,Missing cells,Rows with missing values,Infinite cells,Header repeat rows,Duplicate rows,Duplicate percentage
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,2558,2558,5542,0,5459,0.521
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,0,0,0,1,137229,13.087
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3754,3754,7662,0,2614,0.249
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,36767,36767,82139,0,2,0.000
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,1834,1834,4004,25,97,0.029
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,4921,4921,11133,0,2421,0.231
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3569,3569,7651,0,3278,0.313
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,2277,2277,5371,0,225628,21.518
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,0,0,0,0,17557,1.674
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,4041,4041,8297,33,6121,0.998


In [6]:
# ==============================================================================
# Cell: Numerical Range (Min, Max, Range) Inspection
# ==============================================================================
import os
import glob
import time
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw/CSE-CIC-IDS2018"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

def compute_numerical_ranges(file_path, chunk_size=200000):
    """
    Computes min, max, and numerical range for all numeric columns in a CSV,
    safely handling repeated headers and infinite values.
    """
    fname = os.path.basename(file_path)
    col_mins = None
    col_maxs = None
    total_rows = 0
    first_col = None

    for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
        if first_col is None:
            first_col = chunk.columns[0]
            
        # 1. Filter out repeated header rows
        chunk = chunk[chunk[first_col].astype(str) != first_col]
        total_rows += len(chunk)
        
        # 2. Select numeric columns (exclude Timestamp and Label)
        num_cols = [c for c in chunk.columns if c not in ['Timestamp', 'Label']]
        
        # 3. Convert to numeric and mask infinities
        chunk_num = chunk[num_cols].apply(pd.to_numeric, errors='coerce')
        chunk_num = chunk_num.replace([np.inf, -np.inf], np.nan)
        
        # 4. Compute chunk-level min & max
        c_min = chunk_num.min()
        c_max = chunk_num.max()
        
        if col_mins is None:
            col_mins = c_min
            col_maxs = c_max
        else:
            col_mins = np.minimum(col_mins, c_min)
            col_maxs = np.maximum(col_maxs, c_max)

    df_range = pd.DataFrame({
        'Feature': col_mins.index,
        'Min': col_mins.values,
        'Max': col_maxs.values,
        'Range': (col_maxs - col_mins).values
    })
    
    return df_range, total_rows

# Example: Inspect the first file
sample_file = files[0]
print(f"📊 Analyzing numerical ranges for: {os.path.basename(sample_file)}...")
t0 = time.time()
df_sample_range, rows = compute_numerical_ranges(sample_file)
print(f"Done in {time.time()-t0:.2f}s ({rows:,} rows, {len(df_sample_range)} features)\n")

display(df_sample_range.head(15))


📊 Analyzing numerical ranges for: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv...
Done in 12.11s (1,048,575 rows, 78 features)



,Feature,Min,Max,Range
0,Dst Port,0.0,6.553400e+04,6.553400e+04
1,Protocol,0.0,1.700000e+01,1.700000e+01
2,Flow Duration,0.0,1.200000e+08,1.200000e+08
3,Tot Fwd Pkts,1.0,4.315900e+04,4.315800e+04
4,Tot Bwd Pkts,0.0,6.924100e+04,6.924100e+04
5,TotLen Fwd Pkts,0.0,1.100627e+06,1.100627e+06
6,TotLen Bwd Pkts,0.0,1.010000e+08,1.010000e+08
7,Fwd Pkt Len Max,0.0,1.711000e+03,1.711000e+03
8,Fwd Pkt Len Min,0.0,1.460000e+03,1.460000e+03
9,Fwd Pkt Len Mean,0.0,1.460000e+03,1.460000e+03


In [7]:
# ==============================================================================
# Cell: Consolidated Global Numerical Range Across Entire Dataset
# ==============================================================================
global_mins = None
global_maxs = None
file_range_summaries = {}

t_start = time.time()

for idx, fpath in enumerate(files, 1):
    fname = os.path.basename(fpath)
    print(f"[{idx:02d}/10] Computing ranges for {fname}...", end="", flush=True)
    t0 = time.time()
    
    df_f_range, _ = compute_numerical_ranges(fpath)
    file_range_summaries[fname] = df_f_range.set_index('Feature')
    
    f_min = df_f_range.set_index('Feature')['Min']
    f_max = df_f_range.set_index('Feature')['Max']
    
    if global_mins is None:
        global_mins = f_min
        global_maxs = f_max
    else:
        global_mins = np.minimum(global_mins, f_min)
        global_maxs = np.maximum(global_maxs, f_max)
        
    print(f" ({time.time()-t0:.1f}s)")

# Consolidated table
df_global_ranges = pd.DataFrame({
    'Feature': global_mins.index,
    'Global Min': global_mins.values,
    'Global Max': global_maxs.values,
    'Global Range': (global_maxs - global_mins).values
}).sort_values(by='Global Range', ascending=False).reset_index(drop=True)

print(f"\n✅ All 10 files processed in {time.time()-t_start:.1f}s")
display(df_global_ranges)


[01/10] Computing ranges for Friday-02-03-2018_TrafficForML_CICFlowMeter.csv... (12.4s)
[02/10] Computing ranges for Friday-16-02-2018_TrafficForML_CICFlowMeter.csv... (20.2s)
[03/10] Computing ranges for Friday-23-02-2018_TrafficForML_CICFlowMeter.csv... (13.0s)
[04/10] Computing ranges for Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv... (127.6s)
[05/10] Computing ranges for Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv... (15.9s)
[06/10] Computing ranges for Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv... (13.3s)
[07/10] Computing ranges for Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv... (13.3s)
[08/10] Computing ranges for Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv... (12.8s)
[09/10] Computing ranges for Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv... (11.2s)
[10/10] Computing ranges for Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv... (27.8s)

✅ All 10 files processed in 267.5s


,Feature,Global Min,Global Max,Global Range
0,Fwd IAT Max,-8.282200e+11,9.797810e+11,1.808001e+12
1,Flow IAT Max,-8.282200e+11,9.797810e+11,1.808001e+12
2,Idle Max,0.000000e+00,9.797810e+11,9.797810e+11
3,Flow IAT Min,-9.474050e+11,1.200000e+08,9.475250e+11
4,Fwd IAT Min,-9.474050e+11,1.200000e+08,9.475250e+11
...,...,...,...,...
77,Fwd Pkts/b Avg,0.000000e+00,0.000000e+00,0.000000e+00
78,Dst IP,NaN,NaN,NaN
79,Flow ID,NaN,NaN,NaN
80,Src IP,NaN,NaN,NaN


In [8]:
# ==============================================================================
# Cell: Numerical Anomaly Detection (Zero-Variance, Negative, Extreme Scale)
# ==============================================================================

# 1. Constant / Zero-Variance Features (Range == 0) -> Provide 0 signal to ML
zero_var = df_global_ranges[df_global_ranges['Global Range'] == 0]
print(f"⚠️ Zero-Variance Features ({len(zero_var)} columns - recommend dropping):")
print(zero_var[['Feature', 'Global Min', 'Global Max']].to_string(index=False))

# 2. Features with Negative Values (Min < 0)
negative_cols = df_global_ranges[df_global_ranges['Global Min'] < 0]
print(f"\n⚠️ Features with Negative Values ({len(negative_cols)} columns):")
print(negative_cols[['Feature', 'Global Min', 'Global Max']].to_string(index=False))

# 3. Features with Massive Scales (Max > 10^7) -> Recommend Log / Robust Scaling
extreme_scale = df_global_ranges[df_global_ranges['Global Max'] >= 1e7]
print(f"\n📈 Extreme Scale Features ({len(extreme_scale)} columns - need log/scaling):")
print(extreme_scale[['Feature', 'Global Min', 'Global Max']].head(10).to_string(index=False))


⚠️ Zero-Variance Features (8 columns - recommend dropping):
         Feature  Global Min  Global Max
   Bwd PSH Flags         0.0         0.0
  Bwd Byts/b Avg         0.0         0.0
Bwd Blk Rate Avg         0.0         0.0
  Bwd Pkts/b Avg         0.0         0.0
Fwd Blk Rate Avg         0.0         0.0
  Fwd Byts/b Avg         0.0         0.0
   Bwd URG Flags         0.0         0.0
  Fwd Pkts/b Avg         0.0         0.0

⚠️ Features with Negative Values (11 columns):
          Feature    Global Min   Global Max
      Fwd IAT Max -8.282200e+11 9.797810e+11
     Flow IAT Max -8.282200e+11 9.797810e+11
     Flow IAT Min -9.474050e+11 1.200000e+08
      Fwd IAT Min -9.474050e+11 1.200000e+08
      Fwd IAT Tot -9.190110e+11 1.200000e+08
    Flow Duration -9.190110e+11 1.200000e+08
    Flow IAT Mean -8.282200e+11 1.200000e+08
     Fwd IAT Mean -8.282200e+11 1.200000e+08
      Flow Pkts/s -8.895325e-03 6.000000e+06
Init Fwd Win Byts -1.000000e+00 6.553500e+04
Init Bwd Win Byts -1.000000e

In [9]:
# ==============================================================================
# Cell: Pre-Merge Dataset Compatibility Verification (4 Core Checks)
# ==============================================================================
import os
import glob
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw/CSE-CIC-IDS2018"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

print(f"🔍 Running Pre-Merge Compatibility Checks on {len(files)} files...\n")

# ------------------------------------------------------------------------------
# 1 & 2. Columns & Column Order Check
# ------------------------------------------------------------------------------
file_headers = {}
for f in files:
    fname = os.path.basename(f)
    with open(f, 'r', errors='ignore') as fp:
        file_headers[fname] = [col.strip() for col in fp.readline().strip().split(',')]

canonical_file = files[0]
canonical_fname = os.path.basename(canonical_file)
canonical_cols = file_headers[canonical_fname]

print(f"📌 Base Reference File: {canonical_fname} ({len(canonical_cols)} columns)")

col_diff_report = []
for fname, cols in file_headers.items():
    missing_from_file = set(canonical_cols) - set(cols)
    extra_in_file = set(cols) - set(canonical_cols)
    order_matches = (cols == canonical_cols)
    
    col_diff_report.append({
        'File': fname,
        'Num Columns': len(cols),
        'Matches Base Order': order_matches,
        'Extra Columns': list(extra_in_file) if extra_in_file else 'None',
        'Missing Columns': list(missing_from_file) if missing_from_file else 'None'
    })

df_col_checks = pd.DataFrame(col_diff_report)
print("\n--- Check 1 & 2: Column Set and Order Audit ---")
display(df_col_checks)

# ------------------------------------------------------------------------------
# 3. Data Types & Repeated Header Contamination Check
# ------------------------------------------------------------------------------
type_issues = []
for f in files:
    fname = os.path.basename(f)
    df_sample = pd.read_csv(f, nrows=2000, low_memory=False)
    
    # Check for repeated header rows in sample
    has_repeated_headers = (df_sample['Label'] == 'Label').any()
    
    # Non-numeric columns (excluding expected Timestamp, Label, and IP/FlowID)
    allowed_string_cols = {'Timestamp', 'Label', 'Flow ID', 'Src IP', 'Dst IP'}
    unexpected_object_cols = [
        c for c in df_sample.columns 
        if c not in allowed_string_cols and df_sample[c].dtype == 'object'
    ]
    
    type_issues.append({
        'File': fname,
        'Repeated Headers Present': has_repeated_headers,
        'Unexpected Object Columns': len(unexpected_object_cols),
        'Sample Timestamp': str(df_sample['Timestamp'].iloc[0])
    })

print("\n--- Check 3: Data Type Compatibility & Header Pollution ---")
display(pd.DataFrame(type_issues))

# ------------------------------------------------------------------------------
# 4. Label Consistency Check
# ------------------------------------------------------------------------------
label_summary = []
for f in files:
    fname = os.path.basename(f)
    # Peek unique labels using small chunk reading
    labels_in_file = set()
    for chunk in pd.read_csv(f, chunksize=200000, usecols=['Label'], low_memory=False):
        labels_in_file.update(chunk['Label'].dropna().unique())
        
    # Exclude header repetition 'Label'
    labels_in_file.discard('Label')
    
    label_summary.append({
        'File': fname,
        'Unique Classes Count': len(labels_in_file),
        'Classes Present': sorted(list(labels_in_file))
    })

print("\n--- Check 4: Label Column Audit ---")
display(pd.DataFrame(label_summary))


🔍 Running Pre-Merge Compatibility Checks on 10 files...

📌 Base Reference File: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv (80 columns)

--- Check 1 & 2: Column Set and Order Audit ---


,File,Num Columns,Matches Base Order,Extra Columns,Missing Columns
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,80,True,None,None
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,80,True,None,None
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,80,True,None,None
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,84,False,"[Dst IP, Flow ID, Src IP, Src Port]",None
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,80,True,None,None
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,80,True,None,None
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,80,True,None,None
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,80,True,None,None
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,80,True,None,None
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,80,True,None,None



--- Check 3: Data Type Compatibility & Header Pollution ---


,File,Repeated Headers Present,Unexpected Object Columns,Sample Timestamp
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,False,0,02/03/2018 08:47:38
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,False,0,16/02/2018 08:27:23
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,False,0,23/02/2018 08:18:29
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,False,0,20/02/2018 08:34:07
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,True,0,01/03/2018 08:17:11
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,False,0,15/02/2018 08:25:18
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,False,0,22/02/2018 08:26:03
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,False,0,14/02/2018 08:31:01
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,False,0,21/02/2018 08:33:25
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,False,0,28/02/2018 08:22:13



--- Check 4: Label Column Audit ---


,File,Unique Classes Count,Classes Present
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2,"[Benign, Bot]"
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,3,"[Benign, DoS attacks-Hulk, DoS attacks-SlowHTT..."
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,4,"[Benign, Brute Force -Web, Brute Force -XSS, S..."
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,2,"[Benign, DDoS attacks-LOIC-HTTP]"
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,2,"[Benign, Infilteration]"
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,3,"[Benign, DoS attacks-GoldenEye, DoS attacks-Sl..."
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,4,"[Benign, Brute Force -Web, Brute Force -XSS, S..."
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,3,"[Benign, FTP-BruteForce, SSH-Bruteforce]"
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,3,"[Benign, DDOS attack-HOIC, DDOS attack-LOIC-UDP]"
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,2,"[Benign, Infilteration]"


In [10]:
# ==============================================================================
# Cell: Clean & Merge CSE-CIC-IDS2018 into 'cse-cic-ids2018_merged_raw'
# ==============================================================================
import os
import glob
import time
import pandas as pd
import numpy as np

# Input and Output Paths
DATA_DIR = "../data/raw/CSE-CIC-IDS2018"
OUTPUT_DIR = "../data/interim/CSE-CIC-IDS2018"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_OUTPUT = os.path.join(OUTPUT_DIR, "cse-cic-ids2018_merged_raw.csv")
PARQUET_OUTPUT = os.path.join(OUTPUT_DIR, "cse-cic-ids2018_merged_raw.parquet")

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

# Canonical 80 columns (standard across all 9 normal files)
CANONICAL_COLUMNS = [
    'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts',
    'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
    'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s',
    'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
    'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
    'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
    'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s',
    'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std',
    'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt',
    'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count',
    'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg',
    'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg',
    'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg', 'Subflow Fwd Pkts',
    'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts',
    'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts',
    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min',
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Label'
]

# The 4 leaking/extra identifier columns to discard from Tuesday
DISCARD_COLUMNS = ['Flow ID', 'Src IP', 'Src Port', 'Dst IP']

# 78 columns that must be strictly numeric
NUMERIC_COLUMNS = [c for c in CANONICAL_COLUMNS if c not in ['Timestamp', 'Label']]

print(f"🚀 Starting clean streaming merge of {len(files)} files...")
print(f"📁 Output destination: {OUTPUT_DIR}\n")

# Remove existing file if re-running
if os.path.exists(CSV_OUTPUT):
    os.remove(CSV_OUTPUT)

total_rows_merged = 0
total_headers_dropped = 0
t_start = time.time()

for idx, fpath in enumerate(files, 1):
    fname = os.path.basename(fpath)
    print(f"[{idx:02d}/10] Processing {fname}...", end="", flush=True)
    t0 = time.time()
    file_rows = 0
    file_headers_dropped = 0
    
    # Process in memory-safe chunks
    for chunk in pd.read_csv(fpath, chunksize=300000, low_memory=False):
        
        # ----------------------------------------------------------------------
        # FIX 1: Discard the 4 extra columns if present (Tuesday file)
        # ----------------------------------------------------------------------
        cols_to_drop = [c for c in DISCARD_COLUMNS if c in chunk.columns]
        if cols_to_drop:
            chunk = chunk.drop(columns=cols_to_drop)
            
        # ----------------------------------------------------------------------
        # FIX 2A: Remove repeated header rows (Data Compatibility)
        # ----------------------------------------------------------------------
        header_mask = (chunk['Label'] == 'Label') | (chunk['Dst Port'].astype(str) == 'Dst Port')
        dropped_count = int(header_mask.sum())
        if dropped_count > 0:
            chunk = chunk[~header_mask]
            file_headers_dropped += dropped_count
            total_headers_dropped += dropped_count
            
        # ----------------------------------------------------------------------
        # FIX 2B: Align to canonical order & enforce numeric data types
        # ----------------------------------------------------------------------
        chunk = chunk[CANONICAL_COLUMNS]
        
        # Replace infinities in flow rate columns
        chunk['Flow Byts/s'] = pd.to_numeric(chunk['Flow Byts/s'], errors='coerce').replace([np.inf, -np.inf], np.nan)
        chunk['Flow Pkts/s'] = pd.to_numeric(chunk['Flow Pkts/s'], errors='coerce').replace([np.inf, -np.inf], np.nan)
        
        # Convert all other numeric columns to float32/float64
        chunk[NUMERIC_COLUMNS] = chunk[NUMERIC_COLUMNS].apply(pd.to_numeric, errors='coerce')
        
        # ----------------------------------------------------------------------
        # Write to disk
        # ----------------------------------------------------------------------
        is_first_chunk = (total_rows_merged == 0)
        chunk.to_csv(CSV_OUTPUT, mode='a', index=False, header=is_first_chunk)
        
        chunk_len = len(chunk)
        file_rows += chunk_len
        total_rows_merged += chunk_len
        
    print(f" -> {file_rows:,} rows (dropped {file_headers_dropped} header rows) in {time.time()-t0:.1f}s")

# Convert to Parquet format (preserves schema types and is ~70% smaller)
print("\n📦 Generating optimized Parquet file...")
t_pq = time.time()
pd.read_csv(CSV_OUTPUT, chunksize=500000, low_memory=False)
# Stream CSV to Parquet using pyarrow
import pyarrow.csv as pv
import pyarrow.parquet as pq

table = pv.read_csv(CSV_OUTPUT)
pq.write_table(table, PARQUET_OUTPUT, compression='snappy')
print(f"✅ Parquet generated in {time.time()-t_pq:.1f}s: {PARQUET_OUTPUT}")

print("\n" + "="*80)
print(f"🎉 MERGE COMPLETED SUCCESSFULLY!")
print(f"• Total Rows Merged: {total_rows_merged:,}")
print(f"• Total Repeated Header Rows Purged: {total_headers_dropped}")
print(f"• Discarded Columns (from Tuesday): {DISCARD_COLUMNS}")
print(f"• Final Columns Count: {len(CANONICAL_COLUMNS)}")
print(f"• Total Execution Time: {time.time()-t_start:.1f}s")
print(f"• CSV Saved: {CSV_OUTPUT}")
print(f"• Parquet Saved: {PARQUET_OUTPUT}")
print("="*80)


🚀 Starting clean streaming merge of 10 files...
📁 Output destination: ../data/interim/CSE-CIC-IDS2018

[01/10] Processing Friday-02-03-2018_TrafficForML_CICFlowMeter.csv... -> 1,048,575 rows (dropped 0 header rows) in 47.8s
[02/10] Processing Friday-16-02-2018_TrafficForML_CICFlowMeter.csv... -> 1,048,574 rows (dropped 1 header rows) in 65.3s
[03/10] Processing Friday-23-02-2018_TrafficForML_CICFlowMeter.csv... -> 1,048,575 rows (dropped 0 header rows) in 43.1s
[04/10] Processing Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv... -> 7,948,748 rows (dropped 0 header rows) in 488.2s
[05/10] Processing Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv... -> 331,100 rows (dropped 25 header rows) in 22.1s
[06/10] Processing Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv... -> 1,048,575 rows (dropped 0 header rows) in 36.4s
[07/10] Processing Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv... -> 1,048,575 rows (dropped 0 header rows) in 36.2s
[08/10] Processing Wednesday-14-02-2018_Tr

In [11]:
import pyarrow.parquet as pq
import pandas as pd

# Inspect merged parquet metadata
meta = pq.read_metadata("../data/interim/CSE-CIC-IDS2018/cse-cic-ids2018_merged_raw.parquet")
print(f"✅ Total rows in merged dataset: {meta.num_rows:,}")
print(f"✅ Total columns: {meta.num_columns}")

# Verify label counts
df_labels = pq.read_table("../data/interim/CSE-CIC-IDS2018/cse-cic-ids2018_merged_raw.parquet", columns=['Label']).to_pandas()
print("\n📊 Label Distribution:")
print(df_labels['Label'].value_counts())
print(f"\nSum of all labels: {len(df_labels):,}")


✅ Total rows in merged dataset: 16,232,943
✅ Total columns: 80

📊 Label Distribution:
Label
Benign                      13484708
DDOS attack-HOIC              686012
DDoS attacks-LOIC-HTTP        576191
DoS attacks-Hulk              461912
Bot                           286191
FTP-BruteForce                193360
SSH-Bruteforce                187589
Infilteration                 161934
DoS attacks-SlowHTTPTest      139890
DoS attacks-GoldenEye          41508
DoS attacks-Slowloris          10990
DDOS attack-LOIC-UDP            1730
Brute Force -Web                 611
Brute Force -XSS                 230
SQL Injection                     87
Name: count, dtype: int64

Sum of all labels: 16,232,943


In [12]:
# ==============================================================================
# Cell: Generate & Save Comprehensive Inspection Reports to 'report 2018'
# ==============================================================================
import os
import glob
import time
import pandas as pd
import numpy as np

# Directory paths
DATA_DIR = "../data/raw/CSE-CIC-IDS2018"
REPORT_DIR = "../data/reports/CSE-CIC-IDS2018"
os.makedirs(REPORT_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"📊 Generating inspection reports for {len(files)} files...")
print(f"📁 Destination Folder: {os.path.abspath(REPORT_DIR)}\n")

file_summaries = []
quality_by_file = []
quality_by_column = []
label_records = []

t_start = time.time()

for idx, fpath in enumerate(files, 1):
    fname = os.path.basename(fpath)
    print(f"[{idx:02d}/10] Auditing {fname}...", end="", flush=True)
    t0 = time.time()
    
    file_rows = 0
    file_nulls = 0
    file_infs = 0
    file_header_repeats = 0
    col_nulls = {}
    col_infs = {}
    seen_hashes = set()
    file_dups = 0
    
    for chunk in pd.read_csv(fpath, chunksize=300000, low_memory=False):
        chunk_len = len(chunk)
        file_rows += chunk_len
        
        # 1. Repeated header rows
        first_col = chunk.columns[0]
        header_mask = (chunk[first_col].astype(str) == first_col)
        repeats = int(header_mask.sum())
        file_header_repeats += repeats
        
        # Clean chunk for data analysis
        clean_chunk = chunk[~header_mask]
        
        # 2. Missing values
        nulls = clean_chunk.isna().sum()
        for col, count in nulls[nulls > 0].items():
            col_nulls[col] = col_nulls.get(col, 0) + int(count)
            file_nulls += int(count)
            
        # 3. Infinite values in flow rate columns
        for col in ['Flow Byts/s', 'Flow Pkts/s']:
            if col in clean_chunk.columns:
                if pd.api.types.is_numeric_dtype(clean_chunk[col]):
                    infs = int(np.isinf(clean_chunk[col]).sum())
                else:
                    infs = int(clean_chunk[col].astype(str).str.strip().isin(['Infinity', 'inf', '-Infinity', '-inf']).sum())
                if infs > 0:
                    col_infs[col] = col_infs.get(col, 0) + infs
                    file_infs += infs
                    
        # 4. Duplicate rows (via 64-bit row hash)
        hashes = pd.util.hash_pandas_object(clean_chunk, index=False)
        dups_within = hashes.duplicated()
        dups_cross = hashes.isin(seen_hashes)
        is_dup = dups_within | dups_cross
        file_dups += int(is_dup.sum())
        seen_hashes.update(hashes[~is_dup])
        
        # 5. Label distribution
        if 'Label' in clean_chunk.columns:
            for lbl, cnt in clean_chunk['Label'].value_counts().items():
                label_records.append({'File': fname, 'Label': lbl, 'Count': cnt})

    # File Summary
    cols_count = len(chunk.columns)
    file_summaries.append({'File': fname, 'Rows': file_rows, 'Columns': cols_count})
    
    # Data Quality by File
    dup_pct = round((file_dups / file_rows * 100), 3) if file_rows > 0 else 0
    quality_by_file.append({
        'File': fname,
        'Rows': file_rows,
        'Missing cells': file_nulls,
        'Rows with missing values': file_nulls,
        'Infinite cells': file_infs,
        'Header repeat rows': file_header_repeats,
        'Duplicate rows': file_dups,
        'Duplicate percentage': dup_pct
    })
    
    # Data Quality by Column
    all_affected_cols = set(col_nulls.keys()) | set(col_infs.keys())
    for col in sorted(all_affected_cols):
        quality_by_column.append({
            'File': fname,
            'Column': col,
            'Missing': col_nulls.get(col, 0),
            'Infinite': col_infs.get(col, 0)
        })
        
    print(f" (completed in {time.time()-t0:.1f}s)")

# ------------------------------------------------------------------------------
# Export all 4 CSV Reports to Disk
# ------------------------------------------------------------------------------
df_summary = pd.DataFrame(file_summaries)
df_quality_file = pd.DataFrame(quality_by_file)
df_quality_col = pd.DataFrame(quality_by_column)

df_labels = (
    pd.DataFrame(label_records)
    .groupby(['File', 'Label'])['Count']
    .sum()
    .reset_index()
    .sort_values(by=['File', 'Count'], ascending=[True, False])
)

df_summary.to_csv(os.path.join(REPORT_DIR, "file_summary.csv"), index=False)
df_quality_file.to_csv(os.path.join(REPORT_DIR, "data_quality_by_file.csv"), index=False)
df_quality_col.to_csv(os.path.join(REPORT_DIR, "data_quality_by_column.csv"), index=False)
df_labels.to_csv(os.path.join(REPORT_DIR, "label_distribution_by_file.csv"), index=False)

print("\n" + "="*80)
print(f"✅ ALL 4 REPORT CSVs SAVED IN: {REPORT_DIR}")
print("1. file_summary.csv")
print("2. data_quality_by_file.csv")
print("3. data_quality_by_column.csv")
print("4. label_distribution_by_file.csv")
print(f"Total time elapsed: {time.time()-t_start:.1f}s")
print("="*80)


📊 Generating inspection reports for 10 files...
📁 Destination Folder: c:\Users\himma\Desktop\Projects\AI project\SentinelAI\ml\data\reports\CSE-CIC-IDS2018

[01/10] Auditing Friday-02-03-2018_TrafficForML_CICFlowMeter.csv... (completed in 29.8s)
[02/10] Auditing Friday-16-02-2018_TrafficForML_CICFlowMeter.csv... (completed in 37.8s)
[03/10] Auditing Friday-23-02-2018_TrafficForML_CICFlowMeter.csv... (completed in 34.5s)
[04/10] Auditing Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv... (completed in 483.9s)
[05/10] Auditing Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv... (completed in 25.5s)
[06/10] Auditing Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv... (completed in 34.9s)
[07/10] Auditing Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv... (completed in 33.5s)
[08/10] Auditing Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv... (completed in 31.3s)
[09/10] Auditing Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv... (completed in 30.7s)
[10/10] Auditing Wednesday-

In [13]:
# ==============================================================================
# Cell: Display Generated Reports
# ==============================================================================
REPORT_DIR = "../data/reports/CSE-CIC-IDS2018"

print("📄 1. File Summary:")
display(pd.read_csv(f"{REPORT_DIR}/file_summary.csv"))

print("\n📄 2. Data Quality by File:")
display(pd.read_csv(f"{REPORT_DIR}/data_quality_by_file.csv"))

print("\n📄 3. Data Quality by Affected Column:")
display(pd.read_csv(f"{REPORT_DIR}/data_quality_by_column.csv"))

print("\n📄 4. Label Distribution by File (Top 10 entries):")
display(pd.read_csv(f"{REPORT_DIR}/label_distribution_by_file.csv").head(10))


📄 1. File Summary:


,File,Rows,Columns
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,80
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,84
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,80
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,80
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,80
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,80



📄 2. Data Quality by File:


,File,Rows,Missing cells,Rows with missing values,Infinite cells,Header repeat rows,Duplicate rows,Duplicate percentage
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,2558,2558,5542,0,5459,0.521
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,0,0,0,1,137229,13.087
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3754,3754,7662,0,2614,0.249
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,36767,36767,82139,0,2,0.000
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,1834,1834,4004,25,73,0.022
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,4921,4921,11133,0,2421,0.231
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3569,3569,7651,0,3278,0.313
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,2277,2277,5371,0,225628,21.518
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,0,0,0,0,17557,1.674
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,4041,4041,8297,33,6089,0.993



📄 3. Data Quality by Affected Column:


,File,Column,Missing,Infinite
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,2558,1492
1,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,4050
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,3754,1954
3,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,5708
4,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,36767,22686
5,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,59453
6,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,1834,1085
7,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,2919
8,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,4921,3106
9,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,8027



📄 4. Label Distribution by File (Top 10 entries):


,File,Label,Count
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Benign,762384
1,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Bot,286191
2,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,DoS attacks-Hulk,461912
3,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,Benign,446772
4,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,DoS attacks-SlowHTTPTest,139890
5,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Benign,1048009
6,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Brute Force -Web,362
7,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Brute Force -XSS,151
8,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,SQL Injection,53
9,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Benign,7372557
